# 08 · Platinum-trained XGBoost, evaluated on NEPC / AVPC

Two analyses, each kept within one cohort:

| Analysis | XGBoost trained on | Risk scores | Evaluated on |
|---|---|---|---|
| **ADT-metastatic** | time-to-platinum, ADT-metastatic cohort | nested-CV out-of-fold | time-to-NEPC and time-to-AVPC, ADT-metastatic cohort |
| **Full** | time-to-platinum, full ADT cohort | nested-CV out-of-fold | time-to-NEPC and time-to-AVPC, full ADT cohort |

One model per landmark (0 / 90 / 180 days), full feature set. Fits are always re-run (`FORCE_RERUN = True`).
Only out-of-fold (`cv_oof`) scores are used, so every patient is scored by a model that never saw them.

The two analyses use different patients, so their C-indices are side by side, not a paired difference.
`metastatic_adt` is retrospective (it uses the full observed ADT course), so it tests for signal and is not a deployable model.

In [ ]:
import sys
sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from IPython.display import display
from survival_common import federated_endpoint_transfer as fet
from COMPASS.multivariable_models import local_endpoint_transfer as let

# Analysis name -> cohort key. Each model is trained and evaluated within its own cohort.
COHORTS = {"ADT-metastatic": "metastatic_adt", "Full": "all"}
TRAIN_ENDPOINT = "platinum"
EVAL_ENDPOINTS = ("nepc", "avpc")
N_BOOTSTRAP = 200
SEED = 0

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
OUT_DIR = DATA_ROOT / "local_endpoint_transfer"

cp.N_FOLDS = 5
cp.OUT_OF_FOLD_RISKS = True        # nested-CV held-out score for every patient
cp.FORCE_RERUN = True              # always refit (set False to reuse existing fits)
# XGBoost, full feature set only.
XGB_SPEC = [s for s in cp.MULTIVARIATE_TASK_SPECS if s[:2] == ("xgboost", "both")]

## 1. Fit platinum XGBoost at each landmark and collect OOF risk scores

The full-cohort fit writes to the same output tree as `03_multivariate.ipynb`, so a forced refit replaces those files with the same settings.

In [ ]:
_runs = cp.make_endpoint_runs(["adt"], endpoints=(TRAIN_ENDPOINT,),
                              cohorts=tuple(COHORTS.values()), exclusions=("none",))
RUNS = {name: next(r for r in _runs if r["cohort"] == key) for name, key in COHORTS.items()}

for name, run in RUNS.items():
    failed = [tag for tag, status, _ in cp._run_tasks(run, XGB_SPEC) if status.startswith("FAILED")]
    if failed:
        raise RuntimeError(f"{name}: XGBoost fit failed for\n" + "\n".join(failed))

# OOF scores, keyed by (analysis, landmark). load_oof_scores raises if a file lacks complete cv_oof rows.
OOF = {
    (name, lm): let.load_oof_scores(cp.patient_risk_path(run, "xgboost", lm, "both"), TRAIN_ENDPOINT)
    for name, run in RUNS.items() for lm in run["landmarks"]
}
oof_scores = pd.concat(
    [pd.DataFrame({"analysis": name, "landmark": lm, let.ID_COL: s.index, "risk_score": s.to_numpy()})
     for (name, lm), s in OOF.items()],
    ignore_index=True,
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
oof_scores.to_csv(OUT_DIR / "platinum_xgboost_oof_scores.csv", index=False)
print(f"wrote {OUT_DIR / 'platinum_xgboost_oof_scores.csv'}")
display(oof_scores.groupby(["analysis", "landmark"]).size().rename("patients scored").unstack("landmark"))

## 2. Evaluate the OOF scores on time-to-NEPC and time-to-AVPC

At each landmark, outcomes are incident NEPC / AVPC after the landmark (same outcome construction as the federated transfer in `04`),
restricted to the patients the model scored, which are the patients in its own cohort. 95% CIs come from patient bootstrap.

In [ ]:
local = fet.read_local_frame(DATA_ROOT / "longitudinal_prediction_data_adt.csv", EVAL_ENDPOINTS)
lab_mapping = fet.load_lab_mapping(fet.DEFAULT_LAB_MAPPING)
outcomes = {(ep, lm): let.endpoint_outcomes(local, ep, lm, lab_mapping)
            for ep in EVAL_ENDPOINTS for lm in sorted({lm for _, lm in OOF})}

rows = []
for (name, lm), scores in OOF.items():
    for ep in EVAL_ENDPOINTS:
        # compare() keeps only patients with both an outcome and an OOF score, i.e. this cohort.
        r = let.compare(outcomes[(ep, lm)], {"model": scores}, n_bootstrap=N_BOOTSTRAP, seed=SEED)
        rows.append(r.assign(analysis=name, endpoint=ep.upper(), landmark=lm))

results = (pd.concat(rows, ignore_index=True)
           [["analysis", "endpoint", "landmark", "n", "events",
             "c_index", "c_lo", "c_hi", "mean_auc", "auc_lo", "auc_hi"]]
           .sort_values(["endpoint", "landmark", "analysis"], ignore_index=True))
results.to_csv(OUT_DIR / "platinum_xgboost_on_nepc_avpc.csv", index=False)
print(f"wrote {OUT_DIR / 'platinum_xgboost_on_nepc_avpc.csv'}")

## Results

One row per endpoint x landmark; one column group per analysis. Mean time-dependent AUC is in the CSV.

In [ ]:
def _ci(est, lo, hi):
    return f"{est:.3f} ({lo:.3f}-{hi:.3f})"

table = results.assign(**{
    "n (events)": results["n"].astype(str) + " (" + results["events"].astype(str) + ")",
    "C-index (95% CI)": [_ci(*v) for v in results[["c_index", "c_lo", "c_hi"]].to_numpy()],
})
value_cols = ["n (events)", "C-index (95% CI)"]
table = (table.pivot(index=["endpoint", "landmark"], columns="analysis", values=value_cols)
         .swaplevel(axis=1)
         .reindex(columns=pd.MultiIndex.from_product([list(COHORTS), value_cols])))
display(table)